In [ ]:
# | default_exp transforms/monai/bbox

# Imports

In [ ]:
# | export

from collections.abc import Hashable, Mapping
from typing import Any

import torch
from monai.config import KeysCollection
from monai.transforms import MapTransform

# Transforms

In [ ]:
# | export


class BBoxToMaskd(MapTransform):
    """Create a binary mask from a bounding box and store it under a new key.

    The bounding box is read from ``data[bbox_key]`` and is expected as
    ``(z1, y1, x1, z2, y2, x2)`` in voxel coordinates.  The spatial shape
    is inferred from the single image key provided via ``keys``, which should
    be a channel-first tensor of shape ``(C, D, H, W)``.

    The output mask has shape ``(1, D, H, W)`` with ones inside the bbox and
    zeros elsewhere, stored under ``data[mask_key]``.

    Args:
        keys: exactly one key — the image tensor whose spatial shape
            ``(D, H, W)`` is used to create the mask.
        bbox_key: dictionary key that holds the bounding box.
        mask_key: dictionary key under which to store the new binary mask.
        allow_missing_keys: if ``True``, do not raise if the key is absent
            from the data dict.
    """

    def __init__(
        self,
        keys: KeysCollection,
        bbox_key: str = "bbox",
        mask_key: str = "mask",
        allow_missing_keys: bool = False,
    ) -> None:
        super().__init__(keys, allow_missing_keys=allow_missing_keys)
        assert (
            len(self.keys) == 1
        ), f"BBoxToMaskd expects exactly one key to infer spatial shape, got {len(self.keys)}: {self.keys}"
        self.bbox_key = bbox_key
        self.mask_key = mask_key

    def __call__(self, data: Mapping[Hashable, Any]) -> dict[Hashable, Any]:
        d = dict(data)
        bbox = d[self.bbox_key]
        z1, y1, x1, z2, y2, x2 = (int(b) for b in bbox)

        mask = torch.zeros_like(d[self.keys[0]], dtype=torch.uint8)
        mask[..., z1:z2, y1:y2, x1:x2] = 1

        d[self.mask_key] = mask
        return d

In [ ]:
# --- Test BBoxToMaskd ---

import torch

img = torch.randn(1, 64, 64, 64)

# 1) Basic: mask has correct shape and ones only inside bbox
bbox = [10, 20, 30, 25, 40, 50]
out = BBoxToMaskd(keys="image", bbox_key="bbox")({"image": img, "bbox": bbox})
assert out["mask"].shape == (1, 64, 64, 64), f"Wrong shape: {out['mask'].shape}"
assert out["mask"][0, 10:25, 20:40, 30:50].sum() == (25 - 10) * (40 - 20) * (50 - 30)
assert out["mask"].sum() == out["mask"][0, 10:25, 20:40, 30:50].sum(), "Non-zero values outside bbox"
print(f"[PASS] Basic mask: shape={tuple(out['mask'].shape)}, ones={int(out['mask'].sum())}")

# 2) Custom mask_key
out2 = BBoxToMaskd(keys="image", bbox_key="bbox", mask_key="seg_mask")({"image": img, "bbox": bbox})
assert "seg_mask" in out2 and "mask" not in out2
print("[PASS] Custom mask_key='seg_mask'")

# 3) Original image is unchanged
assert torch.equal(out["image"], img)
print("[PASS] Original image unchanged")

# 4) Bbox at volume boundary (corner case)
bbox_edge = [0, 0, 0, 64, 64, 64]
out4 = BBoxToMaskd(keys="image", bbox_key="bbox")({"image": img, "bbox": bbox_edge})
assert out4["mask"].sum() == 64 * 64 * 64, "Full-volume bbox should be all ones"
print("[PASS] Full-volume bbox")

# 5) Single-voxel bbox
bbox_single = [5, 5, 5, 6, 6, 6]
out5 = BBoxToMaskd(keys="image", bbox_key="bbox")({"image": img, "bbox": bbox_single})
assert out5["mask"].sum() == 1
assert out5["mask"][0, 5, 5, 5] == 1
print("[PASS] Single-voxel bbox")

# 6) Passing multiple keys should fail
try:
    BBoxToMaskd(keys=["image", "other"], bbox_key="bbox")
    assert False, "Should have raised AssertionError"
except AssertionError as e:
    assert "exactly one key" in str(e)
    print(f"[PASS] Multiple keys rejected: {e}")

print("\nAll BBoxToMaskd tests passed!")

[PASS] Basic mask: shape=(1, 64, 64, 64), ones=6000
[PASS] Custom mask_key='seg_mask'
[PASS] Original image unchanged
[PASS] Full-volume bbox
[PASS] Single-voxel bbox
[PASS] Multiple keys rejected: BBoxToMaskd expects exactly one key to infer spatial shape, got 2: ('image', 'other')

All BBoxToMaskd tests passed!


# nbdev

In [ ]:
!nbdev_export